# D16 SQL入门

先在网页完成五题并保存web_practice.sql，再本机复现。选择Python (D Learning)内核，从第一格按顺序运行。SQL在三引号字符串中编写，不能把裸SELECT直接放进Python单元格。

这里留白供你练习。网页SQL和本机SQL是两种数据库环境；本课基础查询通用。


## 1. 本机初始化：准备十名学生

先完成网页的五题，再在本机核对同一批数据。网页使用AlaSQL，本机使用SQLite；今天这五条基础SELECT可在两者运行。
本机只需已有Python和Pandas。每次运行本格都建立一个新的临时内存数据库，不生成数据库文件。
数据库表和Python变量不是同一对象：FROM students引用数据库表；Python中的students是查询得到的DataFrame。
若重启内核，内存数据库会消失；重新运行本格即可恢复样本。


In [2]:
import sqlite3  # Python自带的SQLite接口；SQLite是轻量数据库引擎
import pandas as pd  # pandas用于显示结果和完成后面的对照题
from pathlib import Path  # Path用于拼接文件路径

base_dir = Path("D:/Learning/study/d16")  # 今天的D16文件夹
conn = sqlite3.connect(":memory:")  # conn是connection连接对象；:memory:表示临时内存数据库
init_sql = (base_dir / "data" / "init_students.sql").read_text(encoding="utf-8")  # 读取已准备好的初始化SQL
conn.executescript(init_sql)  # 一次执行建表和填入样本的多条语句；这一步今天可先直接使用

students = pd.read_sql_query("SELECT * FROM students ORDER BY id ASC;", conn)  # 在数据库查询全部列，结果转成DataFrame
print(students)  # 核对十名学生；最左侧0到9是Pandas索引，id列的1到10是学生编号
print(students.shape)  # 预计(10, 5)


   id  class_id name gender  score
0   1         1   小明      M     90
1   2         1   小红      F     95
2   3         1   小军      M     88
3   4         1   小米      F     73
4   5         2   小白      F     81
5   6         2   小兵      M     55
6   7         2   小林      M     85
7   8         3   小新      F     91
8   9         3   小王      M     89
9  10         3   小丽      F     85
(10, 5)


## 2. SQL：选择指定列

只显示id、name、score，并按id升序。先预测10行3列。

预期：10行3列；id从1到10。


In [5]:
# 先写预计行列数，再独立写本节代码。
# SQL需要放在字符串里，使用pd.read_sql_query(sql, conn)运行。
q1_sql = """
SELECT id, name, score  -- SELECT后写需要输出的列，用英文逗号分开
FROM students  -- FROM后是数据库里的表名
ORDER BY id ASC;  -- 按编号升序；分号结束这条语句
"""  # 三个双引号将多行SQL保存为Python字符串
q1 = pd.read_sql_query(q1_sql, conn)  # 执行这一条查询，并把结果读为DataFrame
print(q1)  # 核对查询结果
print(q1.shape)  # 10行3列；id从1到10。


   id name  score
0   1   小明     90
1   2   小红     95
2   3   小军     88
3   4   小米     73
4   5   小白     81
5   6   小兵     55
6   7   小林     85
7   8   小新     91
8   9   小王     89
9  10   小丽     85
(10, 3)


## 3. SQL：两个条件筛选并排序

查询女生且成绩至少90分，只输出id、name、score；按分数降序，同分按id升序。

预期：2行3列：(2,小红,95)、(8,小新,91)。


In [6]:
# 先写预计行列数，再独立写本节代码。
# SQL需要放在字符串里，使用pd.read_sql_query(sql, conn)运行。
q2_sql = """
SELECT id, name, score  -- 结果只显示三个字段
FROM students  -- 从学生表读取记录
WHERE gender = 'F' AND score >= 90  -- 两个条件同时成立；字符串用英文单引号
ORDER BY score DESC, id ASC;  -- 分数从高到低，同分按编号从小到大
"""  # 三个双引号将多行SQL保存为Python字符串
q2 = pd.read_sql_query(q2_sql, conn)  # 执行这一条查询，并把结果读为DataFrame
print(q2)  # 核对查询结果
print(q2.shape)  # 2行3列：(2,小红,95)、(8,小新,91)。


   id name  score
0   2   小红     95
1   8   小新     91
(2, 3)


## 4. SQL：全部成绩降序

不做筛选，显示id、name、score；所有成绩降序，同分编号升序。

预期：10行3列；编号顺序2、8、1、9、3、7、10、5、4、6。


In [7]:
# 先写预计行列数，再独立写本节代码。
# SQL需要放在字符串里，使用pd.read_sql_query(sql, conn)运行。
q3_sql = """
SELECT id, name, score  -- 选择结果列
FROM students  -- 读取所有学生
ORDER BY score DESC, id ASC;  -- 降序优先；第二个键只决定同分者的顺序
"""  # 三个双引号将多行SQL保存为Python字符串
q3 = pd.read_sql_query(q3_sql, conn)  # 执行这一条查询，并把结果读为DataFrame
print(q3)  # 核对查询结果
print(q3.shape)  # 10行3列；编号顺序2、8、1、9、3、7、10、5、4、6。


   id name  score
0   2   小红     95
1   8   小新     91
2   1   小明     90
3   9   小王     89
4   3   小军     88
5   7   小林     85
6  10   小丽     85
7   5   小白     81
8   4   小米     73
9   6   小兵     55
(10, 3)


## 5. SQL：按类别计数

以class_id作为类别，统计每个班级人数，按班级编号升序。

预期：3行2列：1班4人、2班3人、3班3人。


In [8]:
# 先写预计行列数，再独立写本节代码。
# SQL需要放在字符串里，使用pd.read_sql_query(sql, conn)运行。
q4_sql = """
SELECT class_id, COUNT(*) AS student_count  -- 输出班级和人数；AS给结果列起别名
FROM students  -- 统计学生记录
GROUP BY class_id  -- 相同班级编号归为一组，每组统计一次
ORDER BY class_id ASC;  -- 明确按班级编号排序，GROUP BY不代替ORDER BY
"""  # 三个双引号将多行SQL保存为Python字符串
q4 = pd.read_sql_query(q4_sql, conn)  # 执行这一条查询，并把结果读为DataFrame
print(q4)  # 核对查询结果
print(q4.shape)  # 3行2列：1班4人、2班3人、3班3人。


   class_id  student_count
0         1              4
1         2              3
2         3              3
(3, 2)


## 6. SQL：按类别计数并求平均

每班输出班级编号、人数、平均分，并手算一班平均分核对。

预期：3行3列；人数4/3/3，平均分86.5、73.6666667、88.3333333。


In [9]:
# 先写预计行列数，再独立写本节代码。
# SQL需要放在字符串里，使用pd.read_sql_query(sql, conn)运行。
q5_sql = """
SELECT class_id,  -- 分组依据也放入结果，方便辨认各组
       COUNT(*) AS student_count,  -- 统计各组记录数，不要求成绩非空
       AVG(score) AS average_score  -- AVG是average，计算各组非空成绩平均值
FROM students  -- 数据来源为学生表
GROUP BY class_id  -- 按班级聚合
ORDER BY class_id ASC;  -- 结果按班级编号升序
"""  # 三个双引号将多行SQL保存为Python字符串
q5 = pd.read_sql_query(q5_sql, conn)  # 执行这一条查询，并把结果读为DataFrame
print(q5)  # 核对查询结果
print(q5.shape)  # 3行3列；人数4/3/3，平均分86.5、73.6666667、88.3333333。


   class_id  student_count  average_score
0         1              4      86.500000
1         2              3      73.666667
2         3              3      88.333333
(3, 3)


## 7. Pandas对照一：筛选、选列与排序

先关闭SQL参考，用Pandas独立重做q2。它必须实际用loc和sort_values，而不是再次执行SQL来冒充Pandas实现。


In [11]:
# 先写预计行列数，再独立写本节代码。
# SQL需要放在字符串里，使用pd.read_sql_query(sql, conn)运行。
mask = (students["gender"]=="F") & (students["score"]>= 90)
pandas_q2 = students.loc[mask,["id","name","score"]]
pandas_q2 = pandas_q2.sort_values(
    ["score","id"],
    ascending = [False,True]
).reset_index(drop=True)

print(pandas_q2)
pd.testing.assert_frame_equal(pandas_q2, q2)  # 检查两张表的列、行、数值与索引相同

   id name  score
0   2   小红     95
1   8   小新     91


## 8. Pandas对照二：分组计数与平均

独立重做q5。COUNT(*)对应size，AVG(score)对应mean；reset_index将分组索引恢复成普通列。


In [12]:
# 先写预计行列数，再独立写本节代码。
# SQL需要放在字符串里，使用pd.read_sql_query(sql, conn)运行。
pandas_q5 = students.groupby("class_id").agg(
    student_count = ("id","size"),
    average_score = ("score","mean"),
).reset_index()
pandas_q5 = pandas_q5.sort_values("class_id").reset_index(drop=True)
print(pandas_q5)
print(pandas_q5)  # 预计三行三列，各班人数4、3、3
pd.testing.assert_frame_equal(  # 比较SQL和Pandas分组结果
    pandas_q5, q5,  # 两个待比较DataFrame
    check_exact=False, rtol=1e-12, atol=1e-12,  # 平均分允许极小浮点误差
)
print("两道SQL与Pandas对照均通过。")  # 所有比较通过后才会输出



   class_id  student_count  average_score
0         1              4      86.500000
1         2              3      73.666667
2         3              3      88.333333
   class_id  student_count  average_score
0         1              4      86.500000
1         2              3      73.666667
2         3              3      88.333333
两道SQL与Pandas对照均通过。


## 9. 关闭参考，独立检查

从空白重写q2和q5的SQL，再写两份Pandas对应实现。解释筛行、选列、分组和排序的区别。最后填写notes.md。


In [ ]:
# 在这里独立写；先预测结果，再运行核对。
